# Chapter 15 Exercises

This notebook contains exercises covering the advanced deep learning topics discussed in Chapter 15.

---
## Exercise 1: Attention Mechanism

**Topic:** Transformers & Attention

### Problem
Implement a scaled dot-product attention function.

In [ ]:
import torch
import torch.nn.functional as F

def attention(Q, K, V):
    """
    Scaled Dot-Product Attention (see Chapter 15, "Transformers & Attention")
    
    Args:
        Q: Query matrix (batch, seq_len, d_k)
        K: Key matrix (batch, seq_len, d_k)
        V: Value matrix (batch, seq_len, d_v)
    """
    # Get the key dimension (d_k in the paper)
    # This is used for scaling to prevent large dot products
    d_k = Q.size(-1)
    
    # Compute attention scores: Q @ K^T / sqrt(d_k)
    # The scaling by sqrt(d_k) prevents gradient vanishing in large dimensions
    scores = torch.matmul(Q, K.transpose(-2, -1)) / d_k**0.5
    
    # Apply softmax to get attention weights (0-1 range, sum to 1)
    # This determines how much each position attends to others
    weights = F.softmax(scores, dim=-1)
    
    # Final output is weighted sum of values
    return torch.matmul(weights, V), weights

# Test the attention function
Q = torch.randn(2, 3, 64)  # Batch of 2, 3 tokens, 64-dim queries
out, w = attention(Q, Q, Q)  # Self-attention: Q=K=V
print(f"Output shape: {out.shape}")  # (2, 3, 64)

Output shape: torch.Size([2, 3, 64])


---
## Exercise 2: Node Classification with GNN

**Topic:** Graph Neural Networks

### Problem
Build a simple GCN layer for node classification.

In [ ]:
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    """
    Graph Convolutional Network Layer (see Chapter 15, "Graph Neural Networks")
    
    Implements message passing: each node aggregates features from neighbors
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # Linear transformation for node features
        self.linear = nn.Linear(in_ch, out_ch)
    
    def forward(self, x, edge_index):
        """
        Args:
            x: Node features (num_nodes, in_channels)
            edge_index: Graph connectivity (2, num_edges)
        """
        num_nodes = x.size(0)
        
        # Build adjacency matrix from edge_index
        adj = torch.zeros(num_nodes, num_nodes)
        adj[edge_index[0], edge_index[1]] = 1  # edge_index: [sources, targets]
        
        # Add self-loops: each node aggregates its own features too
        adj = adj + torch.eye(num_nodes)
        
        # Normalize: D^-1/2 @ A @ D^-1/2 (symmetric normalization from the GCN paper)
        # This prevents gradient explosion and gives equal importance to all nodes
        deg = adj.sum(1)**-0.5  # D^-1/2 diagonal
        adj = deg.unsqueeze(1) * adj * deg.unsqueeze(0)  # D^-1/2 @ A @ D^-1/2
        
        # Message passing: aggregate neighbor features, then transform
        return adj @ self.linear(x)

# Test the GCN layer
x = torch.randn(4, 16)       # 4 nodes, 16 features each
edges = torch.tensor([[0,1,2],[1,2,3]])  # 3 edges: 0->1, 1->2, 2->3
layer = GCNLayer(16, 32)
print(f"Output shape: {layer(x, edges).shape}")  # (4, 32)

Output shape: torch.Size([4, 32])


---
## Exercise 3: Contrastive Learning Setup

**Topic:** Self-Supervised Learning

### Problem
Implement a basic contrastive learning setup.

In [ ]:
import torch
import torch.nn.functional as F

def nt_xent_loss(z1, z2, temp=0.5):
    """
    NT-Xent (Normalized Temperature-scaled Cross Entropy) Loss
    Used in SimCLR (see Chapter 15, "Self-Supervised Learning")
    
    Goal: Maximize similarity between augmented views of the same image
    """
    # Concatenate embeddings from both views
    # z1, z2 are embeddings of augmented views of the SAME images
    z = torch.cat([z1, z2], dim=0)  # Shape: (2*batch, hidden_dim)
    
    # Compute pairwise similarity matrix, scaled by temperature
    # Higher temp = softer attention, lower temp = sharper focus
    sim = torch.mm(z, z.t()) / temp
    
    # Labels: positive pairs are at positions (i, i) and (i+batch, i+batch)
    # z1[i] should match z2[i], z1[i+batch] should match z2[i+batch]
    labels = torch.arange(len(z1))
    
    # Cross-entropy loss: push positive pairs together, negative pairs apart
    loss = F.cross_entropy(sim, torch.cat([labels, labels]))
    return loss

# Test the contrastive loss
z1 = torch.randn(4, 32)  # Embeddings of view 1 (4 images)
z2 = torch.randn(4, 32)  # Embeddings of view 2 (same 4 images, different augmentation)
loss = nt_xent_loss(z1, z2)
print(f"NT-Xent Loss: {loss:.4f}")

NT-Xent Loss: 25.9385


---

## Solutions Summary

| Exercise | Topic | Key Function |
|----------|-------|-------------|
| 1 | Attention | `attention(Q, K, V)` - scaled dot-product attention |
| 2 | GNN | `GCNLayer` - graph convolution with message passing |
| 3 | Self-Supervised | `nt_xent_loss()` - contrastive learning loss |

## Key Concepts from Chapter 15

1. **Attention**: Q (what to look for), K (what to match), V (what to extract)
2. **GNNs**: Nodes aggregate information from neighbors via edges
3. **Contrastive Learning**: Learn representations by comparing similar/different samples